In [78]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [79]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=30)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,threatAssess,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,sha256,md5,lastFalsePositive,indicator
0,5629499581012209,2025-12-15T16:04:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T11:32:44Z,3.0,78.0,2.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,184.105.247.252
1,5629499578170398,2025-11-26T17:40:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T11:32:44Z,3.0,76.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,167.94.138.161
2,5629499581020718,2025-12-17T16:32:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T11:32:43Z,3.0,79.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,142.93.230.253
3,5629499574014353,2025-10-17T17:10:27Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T11:07:43Z,3.0,70.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,203.155.253.244
4,4246033,2022-08-15T14:50:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T11:07:43Z,3.0,53.0,1.33,...,"[{'id': 140263, 'dateAdded': '2022-08-17T12:32...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.220.101.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,4553627,2024-03-29T13:28:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-09T19:56:49Z,3.0,0.0,3.00,...,"[{'id': 327537, 'dateAdded': '2024-03-29T13:27...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,149.88.27.228
1406,3210,2014-03-11T13:09:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-09-27T17:36:00Z,NaN,NaN,1.08,...,"[{'id': 631, 'dateAdded': '2014-03-13T14:59:33...",rosesci.com,False,False,iSight Partners - Network Conflict-Intel 1044146,NaN,NaN,NaN,NaN,rosesci.com
1407,4488181,2023-12-14T16:33:14Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-09-17T08:57:43Z,1.0,0.0,1.67,...,"[{'id': 291660, 'dateAdded': '2023-12-14T16:32...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.160.111.145
1408,5263298,2025-01-22T18:43:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-26T18:52:15Z,0.0,0.0,0.00,...,"[{'id': 6755399448004157, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-06T00:00:00Z,23.205.105.180


In [80]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src if desired
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')


In [81]:
observed_src_with_tags[observed_src_with_tags['tag_list'].apply(lambda tags: isinstance(tags, list) and any(isinstance(tag, str) and 'Scan' in tag for tag in tags))]

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,hostName,dnsActive,whoisActive,source,address,sha256,md5,lastFalsePositive,indicator,tag_list
155,4554082,2024-03-29T14:31:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T10:07:44Z,1.0,23.0,0.25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,118.193.72.187,"[DHA Splunk API, Web Scanner, cms-soc, OS Splu..."
160,5629499536006284,2025-03-10T13:55:27Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T10:07:43Z,2.0,40.0,2.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.24.28.114,"[DHA Splunk API, Active Scanning, OS Splunk AP..."
219,5629499555060719,2025-06-16T17:42:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T09:47:43Z,4.0,55.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.191.209.74,"[Bulgaria, DHA Splunk API, Web Scanner, OS Spl..."
221,5629499542014593,2025-05-21T13:34:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-29T09:47:43Z,3.0,55.0,0.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.188.206.46,"[Bulgaria, DHA Splunk API, Web Scanner, OS Spl..."
665,6755399465229483,2025-07-28T17:33:44Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-27T09:01:25Z,2.0,26.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,198.235.24.118,"[DHA Splunk API, Web Scanner, OS Splunk API, F..."
677,6755399459061286,2025-06-18T12:15:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-27T09:01:01Z,2.0,74.0,2.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,141.255.164.12,"[Indicator Notice 25168.1, T-Suite, DHA Splunk..."
692,6755399443006882,2025-03-11T16:48:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-27T08:49:04Z,2.0,39.0,2.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.227.189.168,"[Active Scanning, NIH]"
774,6755399442012299,2025-02-21T16:38:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-26T09:00:08Z,3.0,26.0,1.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.183.201.221,"[DHA Splunk API, local file inclusion, directo..."
1102,5629499542008895,2025-05-19T11:39:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-24T08:39:14Z,3.0,49.0,1.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,194.180.49.76,"[DHA Splunk API, Active Scanning, OS Splunk AP..."
1113,5629499547737367,2025-06-04T16:43:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-24T08:39:13Z,3.0,52.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.37.69.113,"[Web-Scanner, DHA Splunk API, OS Splunk API, F..."


In [82]:
# List of tags to count (case-insensitive, strip whitespace)
tags_of_interest = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections"
]

# Normalize tag_list to lowercase for matching
def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag.lower())

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()
    # Add a column to observed_src with the list of matching "Botnet" tags per indicator
    botnet_tags = set(tags_of_interest)
    def extract_botnet_tags(tag_list):
        if not isinstance(tag_list, list):
            return []
        return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in {b.lower() for b in botnet_tags}]

    # Map indicator to its tag_list for efficient lookup
    indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

    observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))
# Display as a DataFrame for readability
pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])

,Tag,Count
0,Scanning,0
1,DDoS,121
2,Spam,0
3,Phishing,20
4,Cryptojacking,0
5,Credential Stuffing,0
6,Ransomware,11
7,Data Theft,32
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [83]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=7):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=30)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_11324\2754178621.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251229.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251228.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251227.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251226.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251225.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251224.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251223.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251222.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251221.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251220.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251219.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251218.csv', 'Z:/HTOC/Data_Analytics/Data/O

'Loaded data from 30 files.'

In [84]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
cutoff_naive = cutoff.tz_convert(None)

# Define known partner names (standalone tags that represent partners)
KNOWN_PARTNERS = {'DHA', 'OS', 'FDA', 'CMS', 'VA', 'HRSA', 'NIH', 'IHS', 'HHS', 'CDC'}

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def get_all_partner_indicators_from_obs(observed_data_df, cutoff_naive):
    """Get ALL indicators that have partners from observed_data_df (no minimum threshold)."""
    if observed_data_df.empty or 'OpDiv' not in observed_data_df.columns:
        return pd.DataFrame()
    
    # Ensure obs_date is datetime
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')
    
    # Filter by recent dates (last 60 days to match wider time window)
    recent_obs = observed_data_df[
        observed_data_df['obs_date'] >= cutoff_naive - timedelta(days=60)
    ].copy()
    
    if recent_obs.empty:
        return pd.DataFrame()
    
    # Group by indicator and count unique OpDiv (partners)
    partner_counts = (
        recent_obs.groupby('indicator')['OpDiv']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count_obs', '<lambda_0>': 'partners_from_obs'})
    )
    
    # Keep ALL indicators with partners (no minimum threshold)
    all_partner_indicators = partner_counts[partner_counts['partner_count_obs'] >= 1].copy()
    
    return all_partner_indicators

def extract_partners_from_tags(observed_src):
    """Extract partner information from ThreatConnect tags."""
    df = observed_src.copy()

    # explode tags.data
    tags_exploded = (
        df[['indicator', 'tags.data']]
          .explode('tags.data')
          .dropna(subset=['tags.data'])
    )

    # normalize all fields in tags.data into flat columns
    tags_norm = pd.json_normalize(tags_exploded['tags.data'])
    tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

    # Replace VA CSOC CTS Splunk with VA Splunk API in tag_name
    tags_norm['tag_name'] = tags_norm['tag_name'].str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)

    # re‑attach indicator
    tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

    # extract partners from tags ending with " Splunk API"
    tags_expanded['partner'] = tags_expanded['tag_name'].map(
        lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
    )

    # aggregate each tag_* field into a list per indicator
    tag_fields = list(tags_norm.columns)
    tag_agg = (
        tags_expanded
          .groupby('indicator', as_index=False)
          .agg({
              **{f: list for f in tag_fields},
              'partner': lambda x: [p for p in dict.fromkeys(x) if p]
          })
          .rename(columns={'partner':'partners_from_tags'})
    )

    # Also extract partners from standalone tags in tag_list (tag_name column)
    def extract_standalone_partners(tag_list):
        """Extract partner names from tag_list that match known partners"""
        if not isinstance(tag_list, list):
            return []
        # Check each tag in the list to see if it matches a known partner name
        found_partners = []
        for tag in tag_list:
            if isinstance(tag, str) and tag.strip() in KNOWN_PARTNERS:
                found_partners.append(tag.strip())
        return found_partners

    # Extract standalone partners from tag_name (which contains the tag_list)
    if 'tag_name' in tag_agg.columns:
        tag_agg['standalone_partners'] = tag_agg['tag_name'].apply(extract_standalone_partners)
        
        # Combine partners from both sources (Splunk API tags and standalone tags)
        def combine_tag_partners(row):
            """Combine partners from Splunk API tags and standalone tags"""
            partners_from_splunk = row.get('partners_from_tags', [])
            partners_standalone = row.get('standalone_partners', [])
            
            # Handle both list and string formats
            if isinstance(partners_from_splunk, str):
                partners_from_splunk = [p.strip() for p in partners_from_splunk.split(',') if p.strip()]
            if not isinstance(partners_from_splunk, list):
                partners_from_splunk = []
            
            # Combine and deduplicate
            all_partners = list(dict.fromkeys(partners_from_splunk + partners_standalone))
            return all_partners
        
        tag_agg['partners_from_tags'] = tag_agg.apply(combine_tag_partners, axis=1)
        
        # Drop the temporary standalone_partners column
        tag_agg = tag_agg.drop(columns=['standalone_partners'], errors='ignore')

    return tag_agg, tag_fields

def combine_partners_from_sources(base_agg, tag_agg, all_partner_indicators):
    """Combine partner information from both observation data and tags."""
    # Merge tag aggregation
    agg_df = base_agg.merge(tag_agg, on='indicator', how='left')
    
    # Merge with all_partner_indicators to get partners from observed_data_df
    if not all_partner_indicators.empty:
        agg_df = agg_df.merge(
            all_partner_indicators[['indicator', 'partners_from_obs', 'partner_count_obs']],
            on='indicator',
            how='left'
        )
    else:
        # Add empty columns if no partner indicators found
        agg_df['partners_from_obs'] = ''
        agg_df['partner_count_obs'] = 0
    
    # Combine partners from all sources
    def combine_all_partners(row):
        """Combine partners from observed_data_df and ThreatConnect tags."""
        obs_partners = row.get('partners_from_obs', '')
        tag_partners = row.get('partners_from_tags', [])
        
        combined = set()
        
        # Add partners from observed_data_df
        if pd.notna(obs_partners) and obs_partners:
            for p in str(obs_partners).split(', '):
                if p.strip():
                    combined.add(p.strip())
        
        # Add partners from tags
        if isinstance(tag_partners, list):
            for p in tag_partners:
                if p and p.strip():
                    combined.add(p.strip())
        elif pd.notna(tag_partners) and tag_partners:
            # Handle case where tag_partners might be a string
            for p in str(tag_partners).split(','):
                if p.strip():
                    combined.add(p.strip())
        
        return ', '.join(sorted(combined)) if combined else ''
    
    # Create combined partners column
    agg_df['partners'] = agg_df.apply(combine_all_partners, axis=1)
    
    # Calculate partner count
    agg_df['partner_count'] = agg_df['partners'].apply(
        lambda x: len([p for p in str(x).split(', ') if p.strip()]) if pd.notna(x) and x else 0
    )
    
    # Clean up temporary columns
    cols_to_drop = ['partners_from_obs', 'partner_count_obs', 'partners_from_tags']
    agg_df = agg_df.drop(columns=[col for col in cols_to_drop if col in agg_df.columns], errors='ignore')
    
    return agg_df

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting enhanced partner extraction pipeline...")

# Step 1: Get ALL indicators with partners from observed_data_df (no filtering)
print("Identifying ALL indicators with partners from observed_data_df...")
all_partner_indicators = get_all_partner_indicators_from_obs(observed_data_df, cutoff_naive)
print(f"Found {len(all_partner_indicators)} indicators with partners from observation data")

# Step 2: Extract partners from ThreatConnect tags
print("Extracting partners from ThreatConnect tags...")
tag_agg, tag_fields = extract_partners_from_tags(observed_src)
print(f"Processed tags for {len(tag_agg)} indicators")

# Step 3: Core aggregation of other columns
print("Performing core aggregation...")
df = observed_src.copy()

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url', 'threatAssessScore', 'calScore'
]

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# Step 4: Combine partners from all sources
print("Combining partners from all sources...")
agg_df = combine_partners_from_sources(base_agg, tag_agg, all_partner_indicators)

# Step 5: Clean up and format list columns
print("Cleaning up list columns...")
def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# Apply cleaning to tag fields (but not partners, which is already a string)
for col in ['group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

print(f"Processing complete! Final dataset has {len(agg_df)} indicators")


display(agg_df)

Starting enhanced partner extraction pipeline...
Identifying ALL indicators with partners from observed_data_df...
Found 2233 indicators with partners from observation data
Extracting partners from ThreatConnect tags...
Processed tags for 1406 indicators
Performing core aggregation...
Combining partners from all sources...
Cleaning up list columns...
Processing complete! Final dataset has 1410 indicators


,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,threatAssessScore,calScore,Botnet,tag_id,tag_name,tag_lastUsed,tag_description,tag_techniqueId,partners,partner_count
0,1-you.njalla.no,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-26T09:31:22Z,0,5.0,...,590,180,[],"465554, 449195, 432131, 267169, 38869, 38539, ...","HHS-CTIR, CVE-2017-0199, Cobalt Strike, Beacon...","2025-11-13T14:00:11Z, 2025-08-11T13:35:06Z, 20...","Adversaries may buy, lease, rent, or obtain in...","T1583, T1098","NIH, VA",2
1,1.54.101.176,6755399460007694,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-23T08:51:53Z,0,1.0,...,196,170,[DDoS],"1469320, 1455870, 505200, 98, 12, 11","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,DHA,1
2,1.zip,4597452,2024-05-10T12:40:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-26T09:27:54Z,0,5.0,...,469,180,[],"264880, 245130, 214752, 39389, 748","UNC5112, TLP: AMBER, Health-ISAC, DarkGate Mal...","2025-07-14T00:00:00Z, 2025-09-04T16:49:49Z, 20...",,,FDA,1
3,101.71.130.99,5629499572136493,2025-10-16T15:41:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-28T10:02:44Z,0,3.0,...,368,180,[],"471298, 35760, 35057, 30479, 23630, 23577, 235...","DHA Splunk API, OS Splunk API, FDA Splunk API,...","2025-11-13T07:08:20Z, 2025-11-12T16:27:29Z, 20...",Observations reported by the HHS Ofice of the ...,,"CMS, DHA, FDA, HHS, NIH, OS",6
4,102.0.5.152,6755399460008259,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-23T08:52:39Z,0,1.0,...,229,180,[DDoS],"1469320, 1455870, 505200, 471298, 30479, 23769...","INDICATOR NOTICE 25178.1, Mr Hamza Group, T-Su...","2025-07-18T13:37:52Z, 2025-10-21T17:48:35Z, 20...",,,"CMS, DHA, VA",3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,www.citiquartz.com,5629499559316557,2025-07-14T15:36:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-23T08:51:54Z,0,5.0,...,363,170,[],"464075, 37189, 37188, 232, 231","Secret Blizzard, UNC638, Venomous Bear, Turla,...","2025-07-14T00:00:00Z, 2025-07-14T00:00:00Z, 20...",,,DHA,1
1406,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-24T13:12:43Z,0,3.0,...,469,180,[],"471298, 461545, 35760, 35752, 35057, 30770, 30...","DHA Splunk API, DeepSeek, OS Splunk API, VA OI...","2025-11-13T07:08:20Z, 2025-01-31T14:04:11Z, 20...",Observations reported by the HHS Ofice of the ...,,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",8
1407,www.help94.top,5629499563052503,2025-08-08T17:35:58Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-27T09:01:21Z,0,5.0,...,567,190,[],"1474942, 1474502, 472692, 451980, 39987, 1502,...","CAMP.24.084, hhs-2025072901, hhs-2025040901, U...","2025-07-30T19:15:12Z, 2025-07-30T19:15:12Z, 20...",,,VA,1
1408,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-24T08:39:39Z,0,4.0,...,363,170,[],"471298, 35760, 35752, 35057, 30770, 30479, 237...","DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...","2025-11-13T07:08:20Z, 2025-11-12T16:27:29Z, 20...",Observations reported by the HHS Ofice of the ...,,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",8


In [85]:
# ──────────────────────────────────────────────────────────────────────────────
# Clean enrichment: filter unsupported types, call providers separately, summarize
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd

COL_PATH = "data.enrichment.data"   # adjust if API changes

# Determine candidate indicators (use 'indicator' if present, else 'summary')
key_col = 'indicator' if 'indicator' in agg_df.columns else 'summary'

# Keep only types that Shodan/VT commonly support; others will be VT-only
supported_types = {
    'Address','IPv4','IPv6','Host','Domain','URL','File','SHA1','SHA256','MD5'
}

candidates = (
    agg_df[[key_col, 'type']]
      .dropna(subset=[key_col])
      .astype({key_col: str})
      .drop_duplicates(subset=[key_col])
)

indicator_values = candidates[key_col].tolist()

display(f"Enriching {len(indicator_values)} indicators (VT; Shodan where supported)...")

enriched_results = []
failed = []

for _, row in candidates.iterrows():
    value = row[key_col]
    typ   = str(row.get('type', '') or '')

    try:
        encoded_value = urllib.parse.quote(value, safe="")

        # Build provider query: always VT; add Shodan only for supported types
        providers = ["VirusTotalV3"]
        if typ in supported_types and typ not in {"URL","File","SHA1","SHA256","MD5"}:
            # Prefer Shodan for network-y types
            providers.append("Shodan")

        q = urllib.parse.urlencode({"type": providers}, doseq=True)
        enrich_url = f"/v3/indicators/{encoded_value}/enrich?{q}"

        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})

        resp = tc.api_request(ro)
        # Many provider errors come back with 200 + status in JSON; don't raise on status
        try:
            data = resp.json()
        except Exception:
            data = {"status": getattr(resp, 'status_code', 'n/a'), "raw": getattr(resp, 'text', None)}

        # Attach key for merge
        data[key_col] = value
        enriched_results.append(data)

    except Exception as e:
        failed.append({key_col: value, "type": typ, "error": str(e)})

# If nothing enriched, carry on with base
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()
else:
    df_enriched = (
        pd.json_normalize(enriched_results)
          .drop_duplicates(subset=[key_col], keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on=key_col, how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[[key_col, COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat[key_col] = exploded[key_col].values

        # ---- Aggregation helpers ---------------------------------------------
        def _flatten_lists(values):
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.unique(flat))
            return flat

        obj_cols = enrich_flat.select_dtypes("object").columns.difference([key_col])
        num_cols = enrich_flat.columns.difference(obj_cols.union({key_col}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
              .groupby(key_col, as_index=False)
              .agg(agg_dict)
        )

        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset=[key_col])
                       .merge(enrich_wide, on=key_col, how="left")
        )

    # Minimal success message
    display(f"Enrichment complete for {recent_tags[key_col].notna().sum()} indicators.")

# Compact failure summary without flooding output
if failed:
    fail_df = pd.DataFrame(failed)
    display(f"{len(failed)} indicators failed enrichment (showing up to 10):")
    display(fail_df.head(10))

# Show preview of key enrichment columns if present
preview_cols = [c for c in [key_col, 'enrich_hostNames', 'enrich_domains', 'enrich_tags', 'enrich_vtMaliciousCount'] if c in recent_tags.columns]

recent_tags.drop(columns=['tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId', 
                          'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count', 
                          'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
                          'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
                          'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
                          'id'], inplace=True, errors='ignore')
                          
display(recent_tags[preview_cols].head(20))


'Enriching 1410 indicators (VT; Shodan where supported)...'

Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host 1-you.njalla.no cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host 1.zip cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host 2-can.njalla.in cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host 3-get.njalla.fo cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress a.nelson@scdatacom.net cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Res

'Enrichment complete for 1410 indicators.'

'86 indicators failed enrichment (showing up to 10):'

,indicator,type,error
0,1-you.njalla.no,Host,"b'{""errCode"":""0x1001"",""message"":""The Host 1-yo..."
1,1.zip,Host,"b'{""errCode"":""0x1001"",""message"":""The Host 1.zi..."
2,2-can.njalla.in,Host,"b'{""errCode"":""0x1001"",""message"":""The Host 2-ca..."
3,3-get.njalla.fo,Host,"b'{""errCode"":""0x1001"",""message"":""The Host 3-ge..."
4,a.nelson@scdatacom.net,EmailAddress,"b'{""errCode"":""0x1001"",""message"":""The EmailAddr..."
5,academy4sc.org,Host,"b'{""errCode"":""0x1001"",""message"":""The Host acad..."
6,air-tite.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host air-..."
7,alphatron.tv,Host,"b'{""errCode"":""0x1001"",""message"":""The Host alph..."
8,alyssa.palmer@southsidechs.org,EmailAddress,"b'{""errCode"":""0x1001"",""message"":""The EmailAddr..."
9,amigoshostel.com,Host,"b'{""errCode"":""0x1001"",""message"":""The Host amig..."


,indicator,enrich_hostNames,enrich_domains,enrich_tags,enrich_vtMaliciousCount
0,1-you.njalla.no,NaN,NaN,NaN,NaN
1,1.54.101.176,None,None,None,0.0
2,1.zip,NaN,NaN,NaN,NaN
3,101.71.130.99,None,None,None,7.0
4,102.0.5.152,[152-5-0-102.r.airtelkenya.com],[airtelkenya.com],[vpn],2.0
5,102.209.18.96,[static-97.veenet.africa],[veenet.africa],[vpn],1.0
6,103.108.229.67,None,None,None,2.0
7,103.114.96.246,None,None,None,1.0
8,103.124.139.170,[ptr-103-124-139-170.gmdp.net.id],[gmdp.net.id],None,2.0
9,103.125.173.75,None,None,None,1.0


In [86]:
recent_tags[recent_tags['indicator'] == 'groundrats.org']

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
1362,groundrats.org,2025-02-13T15:59:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-29T08:46:22Z,0,4.0,48.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [87]:
#recent_tags = recent_tags.head(50)
recent_tags.loc[recent_tags['enrich_vtMaliciousCount'].isnull(), 'enrich_vtMaliciousCount'] = 0
display(recent_tags)

,indicator,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,confidence,...,enrich_city,enrich_country,enrich_domains,enrich_hostNames,enrich_isp,enrich_openPorts,enrich_org,enrich_tags,enrich_vulnerabilities,enrich_vtMaliciousCount
0,1-you.njalla.no,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-26T09:31:22Z,0,5.0,55.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,1.54.101.176,2025-07-02T12:05:31Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-23T08:51:53Z,0,1.0,49.0,...,[Hanoi],[Viet Nam],None,None,[FPT Telecom Company],"[{'transport': 'tcp', 'port': 554, 'data': 'RT...",[FPT Telecom Company],None,None,0.0
2,1.zip,2024-05-10T12:40:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-26T09:27:54Z,0,5.0,77.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,101.71.130.99,2025-10-16T15:41:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-28T10:02:44Z,0,3.0,70.0,...,None,None,None,None,None,None,None,None,None,7.0
4,102.0.5.152,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-23T08:52:39Z,0,1.0,49.0,...,[Nairobi],[Kenya],[airtelkenya.com],[152-5-0-102.r.airtelkenya.com],[Airtel Networks Kenya Limited],"[{'transport': 'udp', 'port': 161, 'product': ...",[Allocated to Airtel Kenya Mobile Broadband an...,[vpn],None,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,www.citiquartz.com,2025-07-14T15:36:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-23T08:51:54Z,0,5.0,77.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1406,www.deepseek.com,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-24T13:12:43Z,0,3.0,63.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1407,www.help94.top,2025-08-08T17:35:58Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-27T09:01:21Z,0,5.0,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1408,www.sthda.com,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-24T08:39:39Z,0,4.0,29.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [88]:
# Count how many indicators are associated with a unique enrich_domains value

# Only proceed if 'enrich_domains' exists in exploded DataFrame
if 'enrich_domains' in exploded.columns:
    # Drop rows where enrich_domains is missing or NaN
    domains_df = exploded.dropna(subset=['enrich_domains'])

    # If enrich_domains is a list, flatten to individual domain rows
    def flatten_domains(row):
        val = row['enrich_domains']
        if isinstance(val, list):
            return [(row['indicator'], d) for d in val if pd.notna(d)]
        elif pd.notna(val):
            return [(row['indicator'], val)]
        return []

    flat = []
    for _, row in domains_df.iterrows():
        flat.extend(flatten_domains(row))

    flat_df = pd.DataFrame(flat, columns=['indicator', 'domain'])

    # Count unique indicators per domain
    domain_counts = (
        flat_df.groupby('domain')['indicator']
        .nunique()
        .reset_index()
        .rename(columns={'indicator': 'indicator_count'})
        .sort_values('indicator_count', ascending=False)
    )

    display(domain_counts)
else:
    display("Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count.")

"Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count."

In [89]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=365)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_11324\1921769075.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251229.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251228.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251227.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251226.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251225.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251224.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251223.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251222.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251221.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251220.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251219.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251218.csv', 'Z:/HTOC/Data_Analytics/Data/Op

In [90]:
# Aggregate observed_data_df by 'indicator', counting unique obs_date occurrences
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)

agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
# Map obs_days_count from agg_by_indicator to recent_tags by indicator, rename to obs_count
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)

,indicator,obs_days_count
0,1-you.njalla.no,28
7,1.54.101.176,1
9,1.zip,1
22,101.71.130.99,75
26,102.0.5.152,17
...,...,...
6340,www.citiquartz.com,2
6343,www.deepseek.com,314
6352,www.help94.top,1
6381,www.sthda.com,252


In [91]:
import pandas as pd
import numpy as np

df = recent_tags.copy()

# ── Severity Binning ────────────────────────────
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

# ── Feature Weights / Params ────────────────────
Weights = {
    "MALICIOUS_WEIGHT": 1.00,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.10,
    "TC_RATING": 0.05,
    "TC_CONFIDENCE": 0.02,
    "TOR_ACTIVITY_WEIGHT": 3.00,  # Bonus for TOR activity
    "CAL_SCORE_WEIGHT": 1.00
}

BOTNET_ACTIONS = {
    'scanning','ddos','spam','phishing','cryptojacking','credential stuffing','ransomware'
}

TOR_ACTIVITY = {
    'tor','tor activity'
}

# ── Utility Functions ────────────────────────────────────────────────
def convert_to_list(val):
    """
    Convert various input types to a list format.
    Handles: None, NaN, lists, tuples, sets, strings (including list representations and comma-separated).
    """
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, set, tuple)):
        return list(val)
    if isinstance(val, str):
        # Try to parse as list if it looks like one
        if val.strip().startswith('[') and val.strip().endswith(']'):
            try:
                import ast
                parsed = ast.literal_eval(val)
                if isinstance(parsed, (list, tuple)):
                    return list(parsed)
            except:
                pass
        # Otherwise treat as comma-separated string
        return [x.strip() for x in val.split(',') if x.strip()]
    return [val] if val else []

# Known feature maximums (absolute, domain-informed)
VT_MAX = 94
MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100

# Policy multipliers
FALSE_POSITIVE_WEIGHT = 0.9         # 10% penalty
BOTNET_MULTIPLIER     = 0.4         # 60% penalty when botnet-related
SCANNER_PENALTY_MULTIPLIER = 0.9    # 30% penalty for scanner/benign-crawler tags
DATA_QUALITY_FLOOR    = 0.85        # at worst 15% reduction for poor completeness

# ── Input caps ──────────────────────────────────
df['obs_count'] = pd.to_numeric(df.get('obs_count', 0), errors='coerce').fillna(0).clip(0, MAX_OBS_REALISTIC)
df['enrich_vtMaliciousCount'] = (
    pd.to_numeric(df.get('enrich_vtMaliciousCount', 0), errors='coerce')
      .fillna(0).clip(0, VT_MAX)
)
df['rating']     = pd.to_numeric(df.get('rating', 0), errors='coerce').fillna(0).clip(0, MAX_RATING)
df['confidence'] = pd.to_numeric(df.get('confidence', 0), errors='coerce').fillna(0).clip(0, MAX_CONFIDENCE)
df['calScore']   = pd.to_numeric(df.get('calScore', 0), errors='coerce').fillna(0).clip(0, 1000)
df['type'] = df.get('type', '').astype(str)

# ── Base additive evidence (no obs additive term) ───────────────
df['malicious_scaled']     = np.power(df['enrich_vtMaliciousCount'], 0.4)
df['malicious_raw_score']  = df['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']
df['continuity_val']       = df['type'].map({
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2, 'Stripped URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3
}).fillna(0)
df['continuity_raw_score'] = df['continuity_val'] * Weights['CONTINUITY_WEIGHT']
df['tc_raw_rating']        = df['rating']     * Weights['TC_RATING']
df['tc_raw_confidence']    = df['confidence'] * Weights['TC_CONFIDENCE']
# Normalize calScore from 0-1000 to 0-1 scale, then apply weight
df['cal_raw_score']        = (df['calScore'] / 1000.0) * Weights['CAL_SCORE_WEIGHT']

# ── TOR Activity detection (check enrich_tags, tag_name, and enrich columns) ───────────────
def has_tor_activity(val):
    # Use the existing TOR_ACTIVITY set for case-insensitive matching
    # Handle None, NaN, or empty values
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    
    # Handle list, set, or tuple
    if isinstance(val, (list, set, tuple)):
        if len(val) == 0:
            return False
        t = " ".join(map(str, val)).lower()
    # Handle string (supports csv-ish strings)
    elif isinstance(val, str):
        if not val or val.strip() == '':
            return False
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
        if t in ['nan', 'none', '']:
            return False
    
    return any(keyword in t for keyword in TOR_ACTIVITY)

# Check for TOR activity in tag_name and enrich_tags only
tor_mask_tag = pd.Series(False, index=df.index)
tor_mask_enrich_tags = pd.Series(False, index=df.index)

if 'enrich_tags' in df.columns:
    tor_mask_enrich_tags = df['enrich_tags'].apply(has_tor_activity)

if 'tag_name' in df.columns:
    # Convert tag_name to list using global convert_to_list function
    tag_name_as_list = df['tag_name'].apply(convert_to_list)
    tor_mask_tag = tag_name_as_list.apply(has_tor_activity)

# Combine TOR masks (OR logic) - only checking tag_name and enrich_tags
tor_flag = (tor_mask_enrich_tags | tor_mask_tag).astype(int)
df['tor_activity_score'] = tor_flag * Weights['TOR_ACTIVITY_WEIGHT']

# ── Boost TOR contribution when VT malicious count is high (>=10) ────────────────
# Fix: Use simple df['enrich_vtMaliciousCount'] instead of nested df.get()
vt_series = pd.to_numeric(df['enrich_vtMaliciousCount'], errors='coerce').fillna(0)
boost_mask = (vt_series >= 10) & tor_flag.astype(bool)

# Apply the boost by doubling the TOR activity score for high VT + TOR indicators
df.loc[boost_mask, 'tor_activity_score'] = df.loc[boost_mask, 'tor_activity_score'] * 2

df['raw_score'] = (
    df['malicious_raw_score'] +
    df['continuity_raw_score'] +
    df['tc_raw_rating'] +
    df['tc_raw_confidence'] +
    df['tor_activity_score'] +
    df['cal_raw_score']
)

# ── Observation penalty (multiplier; linear) ────────────────────
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER   = 0.50  # floor so max reduction is 50% (tune to taste)

obs_frac = df['obs_count'] / MAX_OBS_REALISTIC
df['obs_penalty_multiplier'] = (1.0 - OBS_PENALTY_STRENGTH * obs_frac).clip(OBS_MIN_MULTIPLIER, 1.0)
df['raw_score'] *= df['obs_penalty_multiplier']

# ── Data quality multiplier (light guard) ───────────────────────
needed = ['type','rating','confidence','enrich_vtMaliciousCount']
present_frac = df[needed].notna().sum(axis=1) / len(needed)
df['data_quality_multiplier'] = present_frac.clip(DATA_QUALITY_FLOOR, 1.0)
df['raw_score'] *= df['data_quality_multiplier']

# ── Botnet flag (robust, multi-word, list/string) ───────────────
col = df.get('Botnet', None)
if col is None:
    df['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

# ── False Positive penalty (preview + conditional apply) ────────
if 'falsePositives' in df.columns:
    df['falsePositives'] = pd.to_numeric(df['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df['falsePositives'] > 0
    # Preview: always "what if" penalized
    df['false_positive_raw_score'] = df['raw_score'] * FALSE_POSITIVE_WEIGHT
    # Apply FP only where present
    df.loc[mask_fp, 'raw_score'] = df.loc[mask_fp, 'false_positive_raw_score']
else:
    df['false_positive_raw_score'] = df['raw_score']

# ── Botnet penalty application ──────────────────────────────────
df['botnet_penalty_multiplier'] = np.where(df['botnet_flag'] == 1, BOTNET_MULTIPLIER, 1.0)
df['raw_score'] *= df['botnet_penalty_multiplier']

# ── Scanner penalty (robust tag parse) ──────────────────────────
def has_scanner_tag(val):
    # Standardize all scanner tags to lowercase for case-insensitive matching
    scanners = {
        'scanner', 'masscan', 'zmap', 'shodan', 'censys',
        'active scanning: scanning ip blocks', 'web scanner', 'active scanning'
    }
    
    # Handle None, NaN, or empty values
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    
    # Handle list, set, or tuple
    if isinstance(val, (list, set, tuple)):
        if len(val) == 0:
            return False
        t = " ".join(map(str, val)).lower()
    # Handle string (supports csv-ish strings)
    elif isinstance(val, str):
        if not val or val.strip() == '':
            return False
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
        if t in ['nan', 'none', '']:
            return False
    
    return any(s in t for s in scanners)

# Check both enrich_tags and tag_name columns
scanner_mask_enrich = pd.Series(False, index=df.index)
scanner_mask_tag = pd.Series(False, index=df.index)

if 'enrich_tags' in df.columns:
    scanner_mask_enrich = df['enrich_tags'].apply(has_scanner_tag)

if 'tag_name' in df.columns:
    # Convert tag_name to list using global convert_to_list function
    tag_name_as_list = df['tag_name'].apply(convert_to_list)
    scanner_mask_tag = tag_name_as_list.apply(has_scanner_tag)

# Combine both masks (OR logic - if either has scanner tag, apply penalty)
scanner_mask = scanner_mask_enrich | scanner_mask_tag

df['scanner_penalty_multiplier'] = np.where(scanner_mask, SCANNER_PENALTY_MULTIPLIER, 1.0)
df['raw_score'] *= df['scanner_penalty_multiplier']

# ── Absolute Cap (multipliers are NOT in cap; only additive parts) ───────────
RAW_SCORE_CAP = (
    np.power(VT_MAX, 0.4) * Weights['MALICIOUS_WEIGHT'] +  # <-- fixed to match malicious transform
    float(df['continuity_val'].max() if len(df) else 3) * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (MAX_CONFIDENCE * Weights['TC_CONFIDENCE']) +
    Weights['TOR_ACTIVITY_WEIGHT'] +  # TOR activity bonus
    Weights['CAL_SCORE_WEIGHT']       # CAL score normalized to 0-1 scale
)

# ── Normalize to 0–1000 ─────────────────────────────────────────
df['Threat_Score'] = (1000 * (df['raw_score'] / RAW_SCORE_CAP).clip(0, 1)).round().fillna(0).astype(int)

# ── VT-driven score ceilings / floors ───────────────────────────
if 'enrich_vtMaliciousCount' in df.columns:
    vt_counts = pd.to_numeric(df['enrich_vtMaliciousCount'], errors='coerce').fillna(0)
    low_cap_mask = vt_counts <= 3
    medium_cap_mask = vt_counts.between(5, 10, inclusive='both')
    high_floor_mask = vt_counts >= 13

    # ── TOR Exception Logic: Only apply to indicators that qualify for TOR boost (VT ≥ 10 + TOR) ─────────────
    tor_exception_mask = boost_mask  # Use the same mask as TOR boost requirements
    
    # Ceiling for low VT: keep within low severity band, BUT exclude TOR boost qualifiers
    low_max_score = scoring_bins[1] - 1  # top of "low" bin (e.g., 199)
    # Apply ceiling only to indicators that have low VT AND don't qualify for TOR boost
    low_cap_final_mask = low_cap_mask & ~tor_exception_mask
    df.loc[low_cap_final_mask, 'Threat_Score'] = df.loc[low_cap_final_mask, 'Threat_Score'].clip(upper=low_max_score)

    # Ceiling for medium VT: keep within medium severity band
    medium_max_score = scoring_bins[2] - 1  # top of "medium" bin (e.g., 499)
    #df.loc[medium_cap_mask, 'Threat_Score'] = df.loc[medium_cap_mask, 'Threat_Score'].clip(upper=medium_max_score)

    # Floor for high VT: ensure at least maximum medium score
    df.loc[high_floor_mask, 'Threat_Score'] = df.loc[high_floor_mask, 'Threat_Score'].clip(lower=medium_max_score)

# ── Enhanced Botnet cap with TOR exception ──────────────────────────────────
if 'botnet_flag' in df.columns:
    low_max_score = scoring_bins[1] - 1
    # Create TOR exception mask - only for indicators that qualify for TOR boost (VT ≥ 10 + TOR)
    tor_exception_mask = boost_mask  # Use the same mask as TOR boost requirements
    # Apply botnet cap only to indicators that are botnet-flagged BUT don't qualify for TOR boost
    botnet_cap_final_mask = (df['botnet_flag'] == 1) & ~tor_exception_mask
    df.loc[botnet_cap_final_mask, 'Threat_Score'] = df.loc[botnet_cap_final_mask, 'Threat_Score'].clip(upper=low_max_score)

# Ensure Threat_Score remains integer after clipping
df['Threat_Score'] = df['Threat_Score'].round().astype(int)

# ── Assign Severity bin ─────────────────────────────────────────
df['Severity'] = pd.cut(df['Threat_Score'], bins=scoring_bins, labels=label_bins, right=False)

# ── Explanation (drivers + multipliers) ─────────────────────────
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
    'tor_activity_score': 'TOR activity',
}

def build_explanation(row):
    parts = {
        'malicious_raw_score': float(row.get('malicious_raw_score', 0) or 0),
        'continuity_raw_score': float(row.get('continuity_raw_score', 0) or 0),
        'tc_raw_rating': float(row.get('tc_raw_rating', 0) or 0),
        'tc_raw_confidence': float(row.get('tc_raw_confidence', 0) or 0),
        'tor_activity_score': float(row.get('tor_activity_score', 0) or 0),
    }
    score = float(row.get('Threat_Score', 0) or 0)
    sev = str(row.get('Severity', 'nan'))

    # Top additive drivers (without calculations)
    contrib = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = []
    for k, v in contrib:
        if v != 0:  # Only include non-zero drivers
            label = _NAME_MAP.get(k, k)
            driver_bits.append(label)

    # Multipliers
    botnet_mult  = float(row.get('botnet_penalty_multiplier', 1.0))
    scanner_mult = float(row.get('scanner_penalty_multiplier', 1.0))
    fp_cnt       = int(row.get('falsePositives', 0) or 0)
    tor_score    = float(row.get('tor_activity_score', 0) or 0)

    botnet_note = "Botnet penalty applied." if botnet_mult < 1.0 else "No botnet penalty."
    fp_note     = f"{fp_cnt} false positive tag(s)." if fp_cnt > 0 else "No false positive tags."
    scan_note   = "Scanner penalty applied." if scanner_mult < 1.0 else "No scanner penalty."
    tor_note    = "TOR activity detected." if tor_score > 0 else "No TOR activity."

    drivers_text = "; ".join(driver_bits) if driver_bits else "No significant drivers"

    return (
        f"Severity: {sev}. Top drivers: {drivers_text}. "
        f"{botnet_note} {fp_note} {scan_note} {tor_note} Score: {score:.0f}/1000."
    )

df['Explanation'] = df.apply(build_explanation, axis=1)

# ── De-dupe and show ────────────────────────────────────────────
if 'indicator' in df.columns:
    df.drop_duplicates(subset='indicator', inplace=True)

# ── Rename columns for clarity ──────────────────────────────────
column_rename_map = {
    'indicator': 'Indicator',
    'type': 'Indicator Type',
    'lastObserved': 'Last Observed',
    'enrich_vtMaliciousCount': 'VirusTotal Malicious Score',
    'obs_count': 'Observation Yearly Count',
    'rating': 'ThreatConnect Rating',
    'obs_penalty_multiplier': 'Observation Penalty Multiplier',
    'botnet_flag': 'Botnet Flag',
    'falsePositives': 'False Positives',
    'partners': 'Partners',
    'threatAssessScore': 'ThreatConnect Threat Score',
    'Threat_Score': 'PRISM Score',
    'Severity': 'Severity',
    'Explanation': 'Score Explanation'
}

# Rename columns in dataframe
df.rename(columns=column_rename_map, inplace=True)

# Display with renamed columns
display_columns = [
    'Indicator', 'Indicator Type', 'Last Observed', 'VirusTotal Malicious Score', 
    'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier',
    'Botnet Flag', 'False Positives', 'Partners', 'calScore', 'ThreatConnect Threat Score',
    'PRISM Score', 'Severity', 'Score Explanation'
]

display(df[[col for col in display_columns if col in df.columns]])


,Indicator,Indicator Type,Last Observed,VirusTotal Malicious Score,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,calScore,ThreatConnect Threat Score,PRISM Score,Severity,Score Explanation
0,1-you.njalla.no,Host,2025-12-26 00:00:00+00:00,0.0,28,5.0,0.998466,0,0,"NIH, VA",180,590,137,low,Severity: low. Top drivers: TC confidence; TC ...
1,1.54.101.176,Address,2025-12-04 00:00:00+00:00,0.0,1,1.0,0.999945,1,0,DHA,170,196,136,low,Severity: low. Top drivers: TOR activity; TC c...
2,1.zip,Host,2025-12-08 00:00:00+00:00,0.0,1,5.0,0.999945,0,0,FDA,180,469,172,low,Severity: low. Top drivers: TC confidence; TC ...
3,101.71.130.99,Address,2025-12-29 00:00:00+00:00,7.0,75,3.0,0.995890,0,0,"CMS, DHA, FDA, HHS, NIH, OS",180,368,317,medium,Severity: medium. Top drivers: VT malicious (l...
4,102.0.5.152,Address,2025-12-16 00:00:00+00:00,2.0,17,1.0,0.999068,1,0,"CMS, DHA, VA",180,229,178,low,Severity: low. Top drivers: TOR activity; VT m...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1405,www.citiquartz.com,Host,2025-12-01 00:00:00+00:00,0.0,2,5.0,0.999890,0,0,DHA,170,363,171,low,Severity: low. Top drivers: TC confidence; TC ...
1406,www.deepseek.com,Host,2025-12-29 00:00:00+00:00,0.0,314,3.0,0.982795,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",180,469,140,low,Severity: low. Top drivers: TC confidence; Con...
1407,www.help94.top,Host,2025-12-26 00:00:00+00:00,0.0,1,5.0,0.999945,0,0,VA,190,567,178,low,Severity: low. Top drivers: TC confidence; TC ...
1408,www.sthda.com,Host,2025-12-12 00:00:00+00:00,0.0,252,4.0,0.986192,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",170,363,90,low,Severity: low. Top drivers: TC confidence; Con...


In [92]:
df[df['Indicator'] == '193.163.125.123']

,Indicator,dateAdded,ownerId,ownerName,webLink,Indicator Type,lastModified,False Positives,ThreatConnect Rating,confidence,...,raw_score,Observation Penalty Multiplier,data_quality_multiplier,Botnet Flag,false_positive_raw_score,botnet_penalty_multiplier,scanner_penalty_multiplier,PRISM Score,Severity,Score Explanation
470,193.163.125.123,2025-06-25T17:45:50Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-24T08:44:22Z,0,3.0,54.0,...,3.030304,0.989699,1.0,0,2.727274,1.0,1.0,199,low,Severity: low. Top drivers: VT malicious (log-...


In [110]:
# Save the DataFrame to Excel with only the columns displayed in cell 14
import os
import pandas as pd
from datetime import datetime
from openpyxl.styles import PatternFill

output_dir = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores"
os.makedirs(output_dir, exist_ok=True)

# Create filename
excel_filename = "Threat_Assessment_Scores.xlsx"
excel_path = os.path.join(output_dir, excel_filename)

# Select only the columns to save (using renamed column names)
columns_to_save = ['Indicator', 'Last Observed', 'Indicator Type', 'VirusTotal Malicious Score', 
                   'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier',
                   'Botnet Flag', 'False Positives', 'Partners', 'CAL Score', 'ThreatConnect Score',
                   'PRISM Score', 'Severity', 'Score Explanation']

# Make a copy with only selected columns (only include columns that exist)
columns_to_save = [col for col in columns_to_save if col in df.columns]
df_export = df[columns_to_save].copy()

# Use 'PRISM Score' values for the 'PRISM Score' column in Excel
if 'PRISM Score' in df.columns:
    df_export['PRISM Score'] = df['PRISM Score']
    # Also update Severity to use Severity (Final) if available
    if 'Severity' in df.columns:
        df_export['Severity'] = df['Severity']
# Convert timezone-aware datetime columns to timezone-naive for Excel compatibility
for col in df_export.columns:
    if pd.api.types.is_datetime64_any_dtype(df_export[col]):
        # Check if the column has timezone info
        if df_export[col].dt.tz is not None:
            # Convert to UTC first, then remove timezone info to make it timezone-naive
            df_export[col] = df_export[col].dt.tz_convert('UTC').dt.tz_localize(None)

# Check if file exists and read existing data
if os.path.exists(excel_path):
    df_existing = pd.read_excel(excel_path, engine='openpyxl')
    
    # Normalize existing columns to the latest schema (older files may still use raw names)
    rename_map_existing = {
        old: new for old, new in column_rename_map.items()
        if old in df_existing.columns and new not in df_existing.columns
    }
    if rename_map_existing:
        df_existing.rename(columns=rename_map_existing, inplace=True)
    
    # Add missing columns to existing dataframe with default values
    for col in columns_to_save:
        if col not in df_existing.columns:
            # Set appropriate default values based on column type
            if col == 'Last Observed':
                df_existing[col] = pd.NaT
            elif col in ['VirusTotal Malicious Score', 'Observation Yearly Count', 'ThreatConnect Rating', 
                        'Observation Penalty Multiplier', 'Botnet Flag', 'False Positives', 'PRISM Score', 'ThreatConnect Score']:
                df_existing[col] = 0
            elif col == 'Severity':
                df_existing[col] = ''
            else:
                df_existing[col] = ''
    
    # Ensure both dataframes have the same columns in the same order
    df_existing = df_existing[columns_to_save].copy()
    
    # Count how many indicators will be updated vs added
    existing_indicators = set(df_existing['Indicator'].values)
    new_indicators = set(df_export['Indicator'].values)
    
    indicators_to_add = len(new_indicators - existing_indicators)
    indicators_to_check = existing_indicators & new_indicators
    
    # Find indicators that have actually changed
    indicators_to_update = []
    indicators_unchanged = []
    
    # Set indicator as index for comparison
    df_existing_idx = df_existing.set_index('Indicator').sort_index()
    df_export_idx = df_export.set_index('Indicator').sort_index()
    
    for indicator in indicators_to_check:
        # Compare the rows (excluding the index/indicator column)
        existing_row = df_existing_idx.loc[indicator]
        new_row = df_export_idx.loc[indicator]
        
        # Check if any values have changed
        if not existing_row.equals(new_row):
            indicators_to_update.append(indicator)
        else:
            indicators_unchanged.append(indicator)
    
    # Build the combined dataframe: keep existing unchanged records, update changed ones, add new ones
    # Start with existing records that are unchanged
    df_unchanged = df_existing[df_existing['Indicator'].isin(indicators_unchanged)].copy()
    
    # Add updated records (from new data)
    df_updated = df_export[df_export['Indicator'].isin(indicators_to_update)].copy()
    
    # Add new records (not in existing)
    df_new = df_export[df_export['Indicator'].isin(new_indicators - existing_indicators)].copy()
    
    # Add existing records that are not in new data (preserve historical data)
    df_preserved = df_existing[~df_existing['Indicator'].isin(new_indicators)].copy()
    
    # Combine all parts
    df_combined = pd.concat([df_unchanged, df_updated, df_new, df_preserved], ignore_index=True)
    
    # Final check: remove any duplicates (shouldn't happen, but safety check)
    df_combined = df_combined.drop_duplicates(subset='Indicator', keep='last')
    
    print(f"Existing indicators in new data: {len(indicators_to_check)}")
    print(f"Added (new) indicators: {indicators_to_add}")
    print(f"Updated existing indicators: {len(indicators_to_update)}")
    print(f"Total indicators in sheet: {len(df_combined)}")
else:
    df_combined = df_export.copy()
    # Remove any duplicates in the new data itself
    df_combined = df_combined.drop_duplicates(subset='Indicator', keep='last')
    print(f"Created new file with {len(df_combined)} indicators")

# Save to Excel with severity-based highlighting
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_combined.to_excel(writer, index=False, sheet_name='PRISM Scores')

    # Create and write comparison sheet
    if 'ThreatConnect Score' in df_combined.columns and 'PRISM Score' in df_combined.columns:
        comp_cols = ['Indicator', 'ThreatConnect Score', 'PRISM Score']
        df_comp = df_combined[comp_cols].copy()
        # Ensure numeric
        df_comp['ThreatConnect Score'] = pd.to_numeric(df_comp['ThreatConnect Score'], errors='coerce').fillna(0)
        df_comp['PRISM Score'] = pd.to_numeric(df_comp['PRISM Score'], errors='coerce').fillna(0)
        df_comp['Difference'] = df_comp['PRISM Score'] - df_comp['ThreatConnect Score']
        df_comp.to_excel(writer, index=False, sheet_name='Score Comparison')

    workbook = writer.book
    worksheet = writer.sheets['PRISM Scores']

    

    # Define fills for each severity
    fills = {
        'low': PatternFill(start_color='83de85', end_color='83de85', fill_type='solid'),     # light green
        'medium': PatternFill(start_color='eef084', end_color='eef084', fill_type='solid'),  # light yellow
        'high': PatternFill(start_color='f29953', end_color='f29953', fill_type='solid'),    # light orange
        'critical': PatternFill(start_color='e83f3f', end_color='e83f3f', fill_type='solid') # light red
    }

    for row_idx, severity in enumerate(df_combined['Severity'], start=2):  # start=2 to skip header
        fill = fills.get(str(severity).lower())
        if fill:
            for col_idx in range(1, len(df_combined.columns) + 1):
                worksheet.cell(row=row_idx, column=col_idx).fill = fill

print(f"Saved {len(df_combined)} total indicators with threat assessment scores to {excel_path}")

Existing indicators in new data: 1410
Added (new) indicators: 0
Updated existing indicators: 1410
Total indicators in sheet: 2095


PermissionError: [Errno 13] Permission denied: 'Z:\\HTOC\\Data_Analytics\\Data\\Threat Assessment Scores\\Threat_Assessment_Scores.xlsx'